### Setup

In [0]:
print("dq_check_started")

from pyspark.sql import functions as f
from datetime import datetime

### Use Catalog & Schema

In [0]:
%sql
USE CATALOG marketing_analytics_capstone;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS monitoring;

### Read Silver Data

In [0]:
df = spark.read.table("marketing_analytics_capstone.silver.silver_campaigns")

display(df.limit(10))

### Basic Metrics Calculation

In [0]:
metrics = df.agg(
    f.count("*").alias("total_records"),
    f.sum(f.when(~f.col("dq_pass"), 1).otherwise(0)).alias("bad_records")
).collect()[0]

total_records = metrics["total_records"]
bad_records = metrics["bad_records"]

bad_percentage = bad_records / total_records if total_records > 0 else 0

print(f"Total Records: {total_records}")
print(f"Bad Records: {bad_records}")
print(f"Bad %: {bad_percentage:.2%}")

### Rule-level Failure Breakdown

In [0]:
rule_metrics = df.select(
    f.sum(f.when(~f.col("dq_valid_cost"), 1).otherwise(0)).alias("invalid_cost"),
    f.sum(f.when(~f.col("dq_valid_date"), 1).otherwise(0)).alias("invalid_date"),
    f.sum(f.when(~f.col("dq_valid_impressions"), 1).otherwise(0)).alias("invalid_impressions"),
    f.sum(f.when(~f.col("dq_valid_clicks"), 1).otherwise(0)).alias("invalid_clicks"),
    f.sum(f.when(~f.col("dq_valid_conversion_rate"), 1).otherwise(0)).alias("invalid_conversion_rate")
)

display(rule_metrics)

### Extract Bad Records (Quarantine Dataset)

In [0]:
df_bad = df.filter("dq_pass = false")

display(df_bad)

### Add Failure Reason Column

In [0]:
df_bad = df_bad.withColumn(
    "dq_failure_reason",
    f.concat_ws(",",
        f.when(~f.col("dq_valid_cost"), f.lit("invalid_cost")),
        f.when(~f.col("dq_valid_date"), f.lit("invalid_date")),
        f.when(~f.col("dq_valid_impressions"), f.lit("invalid_impressions")),
        f.when(~f.col("dq_valid_clicks"), f.lit("invalid_clicks")),
        f.when(~f.col("dq_valid_conversion_rate"), f.lit("invalid_conversion_rate"))
    )
)

### Write Quarantine Table

In [0]:
(
    df_bad.write
    .format("delta")
    .mode("append")
    .saveAsTable("marketing_analytics_capstone.silver.silver_campaigns_quarantine")
)

### Threshold Configuration

In [0]:
MAX_BAD_PERCENTAGE = 0.05   # 5%
MAX_BAD_RECORDS = 50        # absolute threshold

### Store Metrics (Monitoring Table)

In [0]:
from pyspark.sql import Row

metrics_row = [Row(
    run_timestamp=datetime.now(),
    total_records=total_records,
    bad_records=bad_records,
    bad_percentage=bad_percentage
)]

spark.createDataFrame(metrics_row).write \
    .mode("overwrite") \
    .saveAsTable("marketing_analytics_capstone.monitoring.data_quality_metrics")

### Final Decision (Pass / Fail Pipeline)

In [0]:
if bad_percentage > MAX_BAD_PERCENTAGE or bad_records > MAX_BAD_RECORDS:
    raise Exception(f"""
    DATA QUALITY CHECK FAILED

    Total Records: {total_records}
    Bad Records: {bad_records}
    Bad Percentage: {bad_percentage:.2%}

    Thresholds:
    Max % Allowed: {MAX_BAD_PERCENTAGE:.2%}
    Max Records Allowed: {MAX_BAD_RECORDS}
    """)
else:
    print(f"""
    DATA QUALITY CHECK PASSED

    Total Records: {total_records}
    Bad Records: {bad_records}
    Bad Percentage: {bad_percentage:.2%}
    """)